In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

# Cargar datos
df = pd.read_csv("../data/raw/german_credit_data.csv")
print(f"✅ Dataset cargado: {df.shape}")

print("\n" + "=" * 60)
print("ANÁLISIS DE OUTLIERS")
print("=" * 60)

# Características numéricas
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Detectar outliers con IQR (Interquartile Range)
print(f"\n📊 Detectando outliers con método IQR:")

outliers_summary = {}
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outliers_summary[col] = len(outliers)
    
    if len(outliers) > 0:
        print(f"   {col}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.2f}%)")

print(f"\n✅ Total de outliers detectados: {sum(outliers_summary.values())}")

✅ Dataset cargado: (1000, 21)

ANÁLISIS DE OUTLIERS

📊 Detectando outliers con método IQR:
   Duration: 70 outliers (7.00%)
   Credit_Amount: 72 outliers (7.20%)
   Age: 23 outliers (2.30%)
   Existing_Credits: 6 outliers (0.60%)
   Dependents: 155 outliers (15.50%)

✅ Total de outliers detectados: 326


In [2]:
print("\n" + "=" * 60)
print("ENCODING DE VARIABLES CATEGÓRICAS")
print("=" * 60)

# Hacer una copia para no modificar el original
df_processed = df.copy()

# Identificar categóricas
categorical_cols = df.select_dtypes(include=['string', 'object']).columns.tolist()
print(f"\nVariables categóricas a codificar: {categorical_cols}")

# Label Encoding (convertir a números)
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col])
    label_encoders[col] = le
    
    print(f"\n{col}:")
    for i, class_name in enumerate(le.classes_):
        print(f"  {class_name} → {i}")

print(f"\n✅ Encoding completado. Shape: {df_processed.shape}")


ENCODING DE VARIABLES CATEGÓRICAS

Variables categóricas a codificar: ['Status', 'Credit_History', 'Purpose', 'Savings', 'Employment', 'Personal_Status', 'Debtors', 'Property', 'Other_Plans', 'Housing', 'Job', 'Telephone', 'Foreign_Worker']

Status:
  A11 → 0
  A12 → 1
  A13 → 2
  A14 → 3

Credit_History:
  A30 → 0
  A31 → 1
  A32 → 2
  A33 → 3
  A34 → 4

Purpose:
  A40 → 0
  A41 → 1
  A410 → 2
  A42 → 3
  A43 → 4
  A44 → 5
  A45 → 6
  A46 → 7
  A48 → 8
  A49 → 9

Savings:
  A61 → 0
  A62 → 1
  A63 → 2
  A64 → 3
  A65 → 4

Employment:
  A71 → 0
  A72 → 1
  A73 → 2
  A74 → 3
  A75 → 4

Personal_Status:
  A91 → 0
  A92 → 1
  A93 → 2
  A94 → 3

Debtors:
  A101 → 0
  A102 → 1
  A103 → 2

Property:
  A121 → 0
  A122 → 1
  A123 → 2
  A124 → 3

Other_Plans:
  A141 → 0
  A142 → 1
  A143 → 2

Housing:
  A151 → 0
  A152 → 1
  A153 → 2

Job:
  A171 → 0
  A172 → 1
  A173 → 2
  A174 → 3

Telephone:
  A191 → 0
  A192 → 1

Foreign_Worker:
  A201 → 0
  A202 → 1

✅ Encoding completado. Shape: (1000, 2

In [3]:
print("\n" + "=" * 60)
print("PREPARACIÓN DE FEATURES Y TARGET")
print("=" * 60)

# Separar features (X) y target (y)
X = df_processed.drop('Target', axis=1)  # Todas las características excepto Target
y = df_processed['Target']               # Variable objetivo

# Target: convertir 1=Good, 2=Bad → 0=Good, 1=Bad (para sklearn)
y = (y - 1).astype(int)  # Ahora 0 y 1

print(f"\n📊 Features (X): {X.shape}")
print(f"   Características: {X.columns.tolist()}")

print(f"\n🎯 Target (y): {y.shape}")
print(f"   Distribución:")
print(f"   - Clase 0 (Good Credit): {(y == 0).sum()} ({(y == 0).sum()/len(y)*100:.1f}%)")
print(f"   - Clase 1 (Bad Credit): {(y == 1).sum()} ({(y == 1).sum()/len(y)*100:.1f}%)")

# Verificar no hay NaN
print(f"\n✅ Values faltantes en X: {X.isnull().sum().sum()}")
print(f"✅ Values faltantes en y: {y.isnull().sum()}")


PREPARACIÓN DE FEATURES Y TARGET

📊 Features (X): (1000, 20)
   Características: ['Status', 'Duration', 'Credit_History', 'Purpose', 'Credit_Amount', 'Savings', 'Employment', 'Installment_Rate', 'Personal_Status', 'Debtors', 'Residence_Since', 'Property', 'Age', 'Other_Plans', 'Housing', 'Existing_Credits', 'Job', 'Dependents', 'Telephone', 'Foreign_Worker']

🎯 Target (y): (1000,)
   Distribución:
   - Clase 0 (Good Credit): 700 (70.0%)
   - Clase 1 (Bad Credit): 300 (30.0%)

✅ Values faltantes en X: 0
✅ Values faltantes en y: 0


In [4]:
from sklearn.model_selection import train_test_split

print("\n" + "=" * 60)
print("TRAIN/TEST SPLIT")
print("=" * 60)

# Split 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📊 Conjunto de ENTRENAMIENTO:")
print(f"   X_train: {X_train.shape}")
print(f"   y_train: {y_train.shape}")
print(f"   - Clase 0: {(y_train == 0).sum()} ({(y_train == 0).sum()/len(y_train)*100:.1f}%)")
print(f"   - Clase 1: {(y_train == 1).sum()} ({(y_train == 1).sum()/len(y_train)*100:.1f}%)")

print(f"\n📊 Conjunto de PRUEBA:")
print(f"   X_test: {X_test.shape}")
print(f"   y_test: {y_test.shape}")
print(f"   - Clase 0: {(y_test == 0).sum()} ({(y_test == 0).sum()/len(y_test)*100:.1f}%)")
print(f"   - Clase 1: {(y_test == 1).sum()} ({(y_test == 1).sum()/len(y_test)*100:.1f}%)")

print(f"\n✅ Split completado. Ambas clases bien representadas (stratified).")


TRAIN/TEST SPLIT

📊 Conjunto de ENTRENAMIENTO:
   X_train: (800, 20)
   y_train: (800,)
   - Clase 0: 560 (70.0%)
   - Clase 1: 240 (30.0%)

📊 Conjunto de PRUEBA:
   X_test: (200, 20)
   y_test: (200,)
   - Clase 0: 140 (70.0%)
   - Clase 1: 60 (30.0%)

✅ Split completado. Ambas clases bien representadas (stratified).
